In [ ]:
import os
import shutil
import random
from pathlib import Path
from collections import defaultdict
from tqdm import tqdm

# ─── CONFIG ─────────────────────────────────────────────────────
SOURCE_ROOT = r"F:\0Defense\Dataset\merged_dataset\merged-dataset-v2"
OUTPUT_ROOT = r"F:\0Defense\Dataset\merged_dataset\merged-dataset-v10"

TRAIN_RATIO = 0.70
VAL_RATIO   = 0.20
TEST_RATIO  = 0.10

SEED = 42
random.seed(SEED)

OLD_SPLITS = ["train", "valid", "test"]  # change to "valid" if needed
# ────────────────────────────────────────────────────────────────


def get_class_from_label(label_path):
    with open(label_path) as f:
        line = f.readline().strip()
    return line.split()[0] if line else None


# Step 1 — Collect ALL samples
print("\n📦 Collecting all samples from merged-dataset-v2...")
class_to_samples = defaultdict(list)

for split in OLD_SPLITS:

    img_dir = Path(SOURCE_ROOT) / "images" / split
    lbl_dir = Path(SOURCE_ROOT) / "labels" / split

    if not img_dir.exists():
        continue

    image_files = list(img_dir.glob("*.*"))

    for img_path in tqdm(image_files, desc=f"Scanning {split}", unit="img"):

        if img_path.suffix.lower() not in [".jpg", ".jpeg", ".png", ".bmp"]:
            continue

        lbl_path = lbl_dir / (img_path.stem + ".txt")
        if not lbl_path.exists():
            continue

        cls = get_class_from_label(lbl_path)
        if cls is not None:
            class_to_samples[int(cls)].append((img_path, lbl_path))


# Report before split
print("\n📊 Samples per class (before split):")
for cls in sorted(class_to_samples.keys()):
    print(f"  Class {cls:>2}: {len(class_to_samples[cls]):>5} images")


# Step 2 — Stratified Split
print("\n✂️ Performing new 70/20/10 split...")

split_data = {"train": [], "valid": [], "test": []}

for cls in tqdm(sorted(class_to_samples.keys()), desc="Splitting classes"):

    samples = class_to_samples[cls]
    random.shuffle(samples)

    n = len(samples)
    n_train = int(n * TRAIN_RATIO)
    n_val   = int(n * VAL_RATIO)

    split_data["train"].extend(samples[:n_train])
    split_data["valid"].extend(samples[n_train:n_train+n_val])
    split_data["test"].extend(samples[n_train+n_val:])


# Step 3 — Write New Dataset
print(f"\n💾 Writing new dataset to {OUTPUT_ROOT} ...")

for split_name, samples in split_data.items():

    img_out = Path(OUTPUT_ROOT) / "images" / split_name
    lbl_out = Path(OUTPUT_ROOT) / "labels" / split_name

    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)

    for img_path, lbl_path in tqdm(samples, desc=f"Writing {split_name}", unit="img"):

        shutil.copy2(img_path, img_out / img_path.name)
        shutil.copy2(lbl_path, lbl_out / lbl_path.name)

    print(f"  {split_name:>5}: {len(samples):>6} images written")


print("\n📋 Final Split Summary:")
for split_name, samples in split_data.items():
    print(f"  {split_name:>5}: {len(samples)} images")

print("\n✅ Done! Dataset successfully re-split with stratified sampling.")